# 18장 실습 — 자원을 세는 법, 그리고 세어도 모르는 것

**대응 이론** 18장 「피지컬 AI 인프라와 남은 문제」 · **실행 등급 A** (CPU / Colab 무료 티어, 전체 약 10초)

마지막 노트북에는 모델도 로봇도 없다. **산술**만 있다.

VLA 프로젝트를 실제로 굴리려면 답해야 하는 질문들이 있다. GPU 가 몇 장 필요한가, 며칠 걸리는가, 메모리에 들어가는가, 디스크가 얼마나 필요한가, 데이터를 얼마나 빨리 읽어야 GPU 가 놀지 않는가. 이 숫자들은 대개 견적서를 쓰기 전에 필요한데, 실측할 수는 없다.

그래서 **추정한다.** 그리고 추정의 값어치는 공식 자체가 아니라 **공개된 실행으로 보정하는 절차**에 있다. 이 노트북의 중심은 거기다 — 계산기가 말이 안 되는 답을 내놓을 때, 그것이 계산기의 고장이 아니라 **가정 중 하나가 틀렸다는 신호**라는 것을 보게 된다.

1. 학습 비용의 골격
2. 공개된 실행으로 보정하기
3. 메모리 — 파라미터 산술이 틀리는 지점
4. 데이터 파이프라인이 진짜 병목일 때
5. 추론 자원 — 로봇 몇 대를 GPU 한 장에
6. 남은 문제

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib

RED, GREY, PINK = "#e32c24", "#c8c8c8", "#f6c9c6"

GPU = {                       # 이름: (bf16 조밀 TFLOPS, 메모리 GB)
    "A100 80GB":  (312,  80),
    "H100 80GB":  (990,  80),
    "L40S 48GB":  (362,  48),
    "RTX 4090":   (165,  24),
}
print("준비 완료")

준비 완료


## 1. 학습 비용의 골격

트랜스포머 학습의 연산량에는 잘 알려진 어림식이 있다. 파라미터 $N$ 개인 모델을 토큰 $D$ 개로 학습하면

<!-- LaTeX: C \approx 6ND \quad \text{FLOPs} -->

6 은 순전파 2 + 역전파 4 에서 온다. 이것을 GPU-시간으로 바꾸려면 하나가 더 필요하다 — **MFU**(model FLOPs utilization), 즉 이론 최대 성능 중 실제로 쓰는 비율이다. 대규모 언어모델 학습에서 흔히 35~50% 를 본다.

<!-- LaTeX: \text{GPU-hours} = \frac{6ND}{\text{TFLOPS}_{\text{peak}} \times \text{MFU} \times 3600} -->

In [2]:
def gpu_hours(N, D, tflops, mfu):
    return 6 * N * D / (tflops * 1e12 * mfu) / 3600

def tokens_from_hours(hours, N, tflops, mfu):
    return hours * 3600 * tflops * 1e12 * mfu / (6 * N)

N = 7.5e9                       # OpenVLA 급 7B
print(f"{'토큰 수':<14}" + "".join(f"{g:>14}" for g in GPU))
print("-" * 70)
for D in (1e10, 5e10, 1e11, 5e11):
    row = "".join(f"{gpu_hours(N, D, GPU[g][0], .35):>13,.0f}h" for g in GPU)
    print(f"{D:<14.0e}{row}")
print(f"\n(N = {N/1e9:.1f}B, MFU 35% 가정)")

토큰 수               A100 80GB     H100 80GB     L40S 48GB      RTX 4090
----------------------------------------------------------------------
1e+10                 1,145h          361h          987h        2,165h
5e+10                 5,723h        1,804h        4,933h       10,823h
1e+11                11,447h        3,608h        9,866h       21,645h
5e+11                57,234h       18,038h       49,329h      108,225h

(N = 7.5B, MFU 35% 가정)


## 2. 공개된 실행으로 보정하기

OpenVLA 는 학습 자원을 공개했다. **A100 64장으로 14일** — 21,504 A100-시간이다. 모델은 7B 이고 학습 데이터는 Open X-Embodiment 에서 추린 궤적 97만 편이다.

공식을 거꾸로 돌려 **이 시간이 몇 토큰을 의미하는지** 구한다.

In [3]:
ACTUAL = 64 * 14 * 24                     # A100-시간
TOK_PER_SAMPLE = 300                      # 이미지 패치 + 지시문 + 액션 토큰, 어림
TRAJ = 970_000

print(f"OpenVLA 공개값: A100 {64}장 × {14}일 = {ACTUAL:,} A100-시간")
print(f"\n{'가정한 MFU':<12}{'설명되는 토큰':>16}{'= 샘플 수':>16}{'궤적당 프레임 50 기준 에폭':>26}")
print("-" * 72)
for mfu in (.20, .35, .50, .65):
    D = tokens_from_hours(ACTUAL, N, GPU["A100 80GB"][0], mfu)
    samples = D / TOK_PER_SAMPLE
    epochs = samples / (TRAJ * 50)
    print(f"{mfu:<12.0%}{D:>16.2e}{samples:>16.2e}{epochs:>25.1f}")

OpenVLA 공개값: A100 64장 × 14일 = 21,504 A100-시간

가정한 MFU              설명되는 토큰          = 샘플 수          궤적당 프레임 50 기준 에폭
------------------------------------------------------------------------
20%                 1.07e+11        3.58e+08                      7.4
35%                 1.88e+11        6.26e+08                     12.9
50%                 2.68e+11        8.95e+08                     18.4
65%                 3.49e+11        1.16e+09                     24.0


In [4]:
print(f"{'궤적당 프레임':<16}{'에폭':>8}{'필요 토큰':>14}{'필요 MFU':>12}")
print("-" * 52)
for fpt in (20, 30, 50, 80):
    for ep in (27,):
        D = TRAJ * fpt * ep * TOK_PER_SAMPLE
        mfu = 6 * N * D / (GPU["A100 80GB"][0] * 1e12 * ACTUAL * 3600)
        flag = "  ← 불가능" if mfu > 1 else ("  ← 비현실적" if mfu > .6 else "")
        print(f"{fpt:<16}{ep:>8}{D:>14.2e}{mfu:>11.0%}{flag}")

궤적당 프레임               에폭         필요 토큰      필요 MFU
----------------------------------------------------
20                    27      1.57e+11        29%
30                    27      2.36e+11        44%
50                    27      3.93e+11        73%  ← 비현실적
80                    27      6.29e+11       117%  ← 불가능


## 3. 메모리 — 파라미터 산술이 틀리는 지점

「LoRA 를 쓰면 메모리가 얼마나 주는가」는 답이 뻔해 보이는 질문이다. 파라미터로 세어 보자.

In [5]:
def mem_params(N, trainable_frac=1.0):
    w = 2 * N                                       # bf16 가중치
    g = 2 * N * trainable_frac                      # 기울기
    o = 12 * N * trainable_frac                     # AdamW: fp32 m, v, master
    return dict(가중치=w, 기울기=g, 옵티마=o)

N7 = 7e9
print(f"{'':<20}{'가중치':>10}{'기울기':>10}{'옵티마':>10}{'합계':>10}")
print("-" * 62)
pred = {}
for tag, frac in [("전량 미세조정", 1.0), ("LoRA r=32 (~1%)", .01)]:
    d = mem_params(N7, frac); tot = sum(d.values()) / 1e9
    pred[tag] = tot
    print(f"{tag:<20}" + "".join(f"{v/1e9:>9.0f}G" for v in d.values()) + f"{tot:>9.0f}G")
print(f"\n파라미터 산술이 말하는 절감 배율: {pred['전량 미세조정']/pred['LoRA r=32 (~1%)']:.1f}배")
print(f"OpenVLA 가 실제로 보고한 값(배치 16): 전량 163.3 GB → LoRA 59.7 GB = {163.3/59.7:.1f}배")
print(f"차이: 활성값(activation) {163.3-pred['전량 미세조정']:.0f} GB 가 양쪽에 그대로 남는다")

                           가중치       기울기       옵티마        합계
--------------------------------------------------------------
전량 미세조정                    14G       14G       84G      112G
LoRA r=32 (~1%)            14G        0G        1G       15G

파라미터 산술이 말하는 절감 배율: 7.5배
OpenVLA 가 실제로 보고한 값(배치 16): 전량 163.3 GB → LoRA 59.7 GB = 2.7배
차이: 활성값(activation) 51 GB 가 양쪽에 그대로 남는다


## 4. 데이터 파이프라인이 진짜 병목일 때

로봇 데이터는 영상이다. 저장과 대역폭을 세어 본다.

In [6]:
CAM, H, W, HZ, SEC = 2, 224, 224, 15, 20
raw_mb = CAM * H * W * 3 * HZ * SEC / 1e6
jpg_mb = raw_mb * .08                                  # JPEG 압축 후 어림
print(f"에피소드 1편 (카메라 {CAM}대, {H}x{W}, {HZ}Hz, {SEC}초)")
print(f"  원본 {raw_mb:.0f} MB   JPEG {jpg_mb:.1f} MB")
print(f"\n{'데이터셋 규모':<22}{'JPEG 저장':>14}{'원본 환산':>14}")
print("-" * 50)
for n, lbl in [(60_000, "BridgeData V2 급"), (107_000, "RoboMIND 급"),
               (970_000, "OpenVLA 학습분"), (1_000_000, "AgiBot World Beta 급")]:
    print(f"{lbl+f' ({n//1000}k편)':<22}{n*jpg_mb/1e6:>12.1f} TB{n*raw_mb/1e6:>12.1f} TB")

print(f"\n{'GPU 수':<10}{'배치/GPU':>10}{'스텝 시간':>11}{'필요 대역폭':>14}{'필요 JPEG 디코드':>18}")
print("-" * 66)
for gpus in (8, 32, 64, 256):
    bs, step = 16, .35
    imgs_s = gpus * bs / step * CAM
    bw = imgs_s * (H * W * 3) / 1e9
    print(f"{gpus:<10}{bs:>10}{step:>10.2f}s{bw:>12.1f} GB/s{imgs_s:>15,.0f} 장/s")

에피소드 1편 (카메라 2대, 224x224, 15Hz, 20초)
  원본 90 MB   JPEG 7.2 MB

데이터셋 규모                      JPEG 저장         원본 환산
--------------------------------------------------
BridgeData V2 급 (60k편)         0.4 TB         5.4 TB
RoboMIND 급 (107k편)             0.8 TB         9.7 TB
OpenVLA 학습분 (970k편)            7.0 TB        87.6 TB
AgiBot World Beta 급 (1000k편)         7.2 TB        90.3 TB

GPU 수         배치/GPU      스텝 시간        필요 대역폭       필요 JPEG 디코드
------------------------------------------------------------------
8                 16      0.35s         0.1 GB/s            731 장/s
32                16      0.35s         0.4 GB/s          2,926 장/s
64                16      0.35s         0.9 GB/s          5,851 장/s
256               16      0.35s         3.5 GB/s         23,406 장/s


## 5. 추론 자원 — 로봇 몇 대를 GPU 한 장에

배포 쪽 산술은 더 단순하다. 정책 한 번 도는 데 걸리는 시간과 로봇이 요구하는 주기만 있으면 된다.

In [7]:
def robots_per_gpu(latency_ms, policy_hz, K=1, util=.8):
    # 한 로봇이 초당 요구하는 추론 횟수 = policy_hz / K
    per_robot = policy_hz / K
    capacity = 1000 / latency_ms * util
    return capacity / per_robot

print(f"{'시나리오':<30}{'지연':>9}{'청크':>7}{'로봇/GPU':>11}")
print("-" * 58)
cases = [("서버 H100, 3B 모델", 36, 1), ("서버 H100, 3B 모델", 36, 8),
         ("서버 A100, 7B 모델", 240, 1), ("서버 A100, 7B 모델", 240, 8),
         ("엣지 Thor, 3B 모델", 93, 1), ("엣지 Thor, 3B 모델", 93, 8)]
for name, lat, K in cases:
    print(f"{name:<30}{lat:>7}ms{K:>7}{robots_per_gpu(lat, 10, K):>11.1f}")

print("\n같은 계산을 뒤집으면 — 로봇 100대를 10Hz 로 돌리려면")
for name, lat, K in [("H100, 3B", 36, 8), ("A100, 7B", 240, 8)]:
    need = 100 * (10 / K) / (1000 / lat * .8)
    print(f"  {name:<14} GPU {need:>5.1f}장")

시나리오                                 지연     청크     로봇/GPU
----------------------------------------------------------
서버 H100, 3B 모델                     36ms      1        2.2
서버 H100, 3B 모델                     36ms      8       17.8
서버 A100, 7B 모델                    240ms      1        0.3
서버 A100, 7B 모델                    240ms      8        2.7
엣지 Thor, 3B 모델                     93ms      1        0.9
엣지 Thor, 3B 모델                     93ms      8        6.9

같은 계산을 뒤집으면 — 로봇 100대를 10Hz 로 돌리려면
  H100, 3B       GPU   5.6장
  A100, 7B       GPU  37.5장
